In [1]:
import pandas as pd

# Read the TSV file with specific columns
columns_to_read = ['rowid', 'Scan', 'Annotation','Score','ProtsAll', 'AnnotationOther', 'ChargeOther','ScoreOther','ProtsAllOther',
                   'MinNTermAdd', 'minNTermSubtract', 'MinCTermAdd', 'minCTermSubtract']
mismatch_df = pd.read_csv('PRISMAL_REPROCESS_PA_UniproKB_Normal123_mismatched.tsv', sep='\t', usecols=columns_to_read)
matched_df = pd.read_csv('PRISMAL_REPROCESS_PA_UniproKB_Normal123_matched.tsv', sep='\t', usecols=columns_to_read)

# Display basic information about the dataset
print(f"Dataset shape: {mismatch_df.shape}")
print(f"Columns: {list(mismatch_df.columns)}")
print("\nFirst few rows:")
mismatch_df.head()

Dataset shape: (5476, 13)
Columns: ['rowid', 'Scan', 'Annotation', 'Score', 'ProtsAll', 'AnnotationOther', 'ChargeOther', 'ScoreOther', 'ProtsAllOther', 'MinNTermAdd', 'minNTermSubtract', 'MinCTermAdd', 'minCTermSubtract']

First few rows:


,rowid,Scan,Annotation,Score,ProtsAll,AnnotationOther,ChargeOther,ScoreOther,ProtsAllOther,MinNTermAdd,minNTermSubtract,MinCTermAdd,minCTermSubtract
0,1,21453,SGKEEALC+57.021Q+0.984LQEENR,11.572654,"(sp|Q6DT37|MRCKG_HUMAN,1,3,0,0)",+42.011MM+15.995WSNFFLQEENR,2,95.855103,"(sp|Q9BXT2|CCG6_HUMAN,0,0,0,0);(tr|A6NFR2|A6NF...",0,0,0,0
1,2,253,EDEEEDERRN+0.984TTVGGAEK,-4.443470,"(XXX_sp|Q93033|IGSF2_HUMAN,2,8,0,0)",C+39.995ASNSESDNAAC+57.021EILLAEK,2,86.569397,"(sp|A0A087WXM9|MEIKN_HUMAN,0,0,0,0);(tr|A0A0G2...",0,0,0,0
2,3,20415,NNYDDYRM+15.995DWL,14.269265,"(sp|P04004|VTNC_HUMAN,16,7,18,4);(tr|H0YJW9|H0...",NNYKCCQSENCSK,2,84.479897,"(sp|Q96CE8|T4S18_HUMAN,31,4,0,0);(tr|C9J6Q4|C9...",31,4,0,0
3,4,1162,+43.006TLIQDC+57.021KDTL,39.583645,"(sp|O43511-2|S26A4_HUMAN,12,7,20,3);(sp|O43511...",KEINDGKECTL,2,82.735199,"(sp|A8MXE2|B3GT9_HUMAN,17,1,11,4);(tr|A0A8I5KV...",17,1,11,4
4,5,1763,SLLSEDTHPC+57.021D,18.082119,"(sp|Q96BY6-3|DOC10_HUMAN,4,-1,29,-1);(sp|Q96BY...",TVISEEGIGC+119.004F,2,81.504501,"(sp|O15172|PSPHL_HUMAN,14,-1,5,-1)",14,-1,5,-1


create a new tsv named mismatch_summary.tsv with the following columns: peptide, peptide_demod, reason_mismatch,'MinNTermAdd', 'minNTermSubtract', 'MinCTermAdd', 'minCTermSubtract' where go through each column of mismatch_df, the peptide in new csv is  AnnotationOther, and peptide_demod is AnnotationOther but get rid of the modifications, for example "+42.011FRNF+28.011GGL" is FRNFGGLLGPMDEPVGMQKWGK. and 'MinNTermAdd', 'minNTermSubtract', 'MinCTermAdd', 'minCTermSubtract' columns are the same from mismatch df. leave reason_mismatch empty for now

In [2]:
import re

def remove_modifications(peptide_string):
    """Remove modification annotations from peptide sequence"""
    if pd.isna(peptide_string):
        return ''
    # Remove modification patterns like +42.011, -17.027, +28.011, etc.
    cleaned = re.sub(r'[+-][\d.]+', '', str(peptide_string))
    return cleaned

# Create the new dataframe with required columns
def extract_protein_ids(prots_string):
    """Extract protein IDs from the ProtsAll format"""
    if pd.isna(prots_string):
        return ''
    # Extract protein IDs from format like (sp|Q9H0U3-2|MAGT1_HUMAN,1,13,0,0)
    protein_ids = re.findall(r'\|([A-Z0-9-]+)\|', str(prots_string))
    return ';'.join(protein_ids) if protein_ids else ''

def extract_fist_protein_id(prots_string):
    """Extract the first protein ID from the ProtsAll format"""
    if pd.isna(prots_string):
        return ''
    match = re.search(r'\|([A-Z0-9-]+)\|', str(prots_string))
    if match:
        return match.group(1).split('-')[0]
    return ''

summary_df = pd.DataFrame({
    'scan': matched_df['Scan'],
    'DB_search_score': matched_df['Score'],
    'precursor_score': matched_df['ScoreOther'],
    'DB_proteins': matched_df['ProtsAll'].apply(lambda x: extract_fist_protein_id(x) if 'Annotation' in matched_df.columns else ''),
    'precursor_proteins': matched_df['ProtsAllOther'].apply(lambda x: extract_fist_protein_id(x) if 'AnnotationOther' in matched_df.columns else ''),
    'precursor_better': (matched_df['Score'] < matched_df['ScoreOther']).astype(int),
    'charge': matched_df['ChargeOther'],
    'peptide': matched_df['AnnotationOther'],
    'peptide_demod': matched_df['AnnotationOther'].apply(remove_modifications),
    'peptide_length': matched_df['AnnotationOther'].apply(remove_modifications).apply(len),
    'reason_mismatch': '',
    'MinNTermAdd': matched_df['MinNTermAdd'],
    'minNTermSubtract': matched_df['minNTermSubtract'],
    'MinCTermAdd': matched_df['MinCTermAdd'],
    'minCTermSubtract': matched_df['minCTermSubtract']
})

# Save to TSV file
summary_df.to_csv('matched_summary.tsv', sep='\t', index=False)

print(f"Created mismatch_summary.tsv with {len(summary_df)} rows")
print("\nFirst few rows of the summary:")
summary_df.head()

Created mismatch_summary.tsv with 14490 rows

First few rows of the summary:


,scan,DB_search_score,precursor_score,DB_proteins,precursor_proteins,precursor_better,charge,peptide,peptide_demod,peptide_length,reason_mismatch,MinNTermAdd,minNTermSubtract,MinCTermAdd,minCTermSubtract
0,3792,109.810234,109.809998,P0C264,P0C264,0,2,SPPLAVLDFLGDDWGLQGNR,SPPLAVLDFLGDDWGLQGNR,20,,0,0,0,0
1,24866,108.666397,108.666000,Q9UKG4,Q9UKG4,0,2,SNADLTTLMHNENLNGVPSITNPIK,SNADLTTLMHNENLNGVPSITNPIK,25,,0,-1,0,0
2,6352,106.826088,106.825996,P61224,P61224,0,2,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,,0,0,5,21
3,6353,103.259895,103.260002,P61224,P61224,1,2,KQVEVDAQQC+57.021MLEILDTAGTEQ,KQVEVDAQQCMLEILDTAGTEQ,22,,0,0,5,21
4,3790,101.337784,101.337997,P0C264,P0C264,1,2,SPPLAVLDFLGDDWGLQGNR,SPPLAVLDFLGDDWGLQGNR,20,,0,0,0,0


use quick indexing methods (with progress bar) to read from Human_UP000005640_2025_05_29_with_isoforms_CC_with_decoys.fasta, filter out the peptides if the peptide_demod is not in the fasta file, print the new summary_df head and number of columns 

In [3]:
# from tqdm import tqdm

# # Read FASTA file and create a set of all protein sequences
# print("Reading FASTA file...")
# fasta_sequences = set()

# with open('data/Human_UP000005640_2025_05_29_with_isoforms_CC_with_decoys.fasta', 'r') as f:
#     current_seq = []
#     for line in tqdm(f, desc="Loading FASTA"):
#         line = line.strip()
#         if line.startswith('>'):
#             # Save previous sequence if exists
#             if current_seq:
#                 fasta_sequences.add(''.join(current_seq))
#                 current_seq = []
#         else:
#             current_seq.append(line)
#     # Add the last sequence
#     if current_seq:
#         fasta_sequences.add(''.join(current_seq))

# print(f"Loaded {len(fasta_sequences)} protein sequences from FASTA file")

# # Create a combined string for faster substring searching (optional optimization)
# # For very large FASTA files, searching individual sequences might be faster
# combined_fasta = ''.join(fasta_sequences)

# # Filter peptides that are found in the FASTA file
# print("\nFiltering peptides...")
# peptides_found = []

# for idx, row in tqdm(summary_df.iterrows(), total=len(summary_df), desc="Checking peptides"):
#     peptide_demod = row['peptide_demod']
#     # Check if peptide is a substring of the combined FASTA sequences
#     if peptide_demod in combined_fasta:
#         peptides_found.append(True)
#     else:
#         peptides_found.append(False)

# # Filter the dataframe
# summary_df = summary_df[peptides_found].reset_index(drop=True)

# print(f"\nFiltered summary_df shape: {summary_df.shape}")
# print(f"Number of columns: {len(summary_df.columns)}")
# print("\nFirst few rows:")
# print(summary_df.head())

get the identified_peptide_list as a set of tuples with (peptide_demod, mass_shift, charge), now get them all from matched_df (already read earlier), where the get the peptide demod similar to mismatch df, get the mass shift as +digits or -digits from the peptide modified, and get charge from the df column 'Charge'

In [4]:
# from tqdm import tqdm

# # Extract peptide information from matched_df
# print("Processing matched_df to create identified_peptide_list...")

# identified_peptide_list = set()

# for idx, row in tqdm(matched_df.iterrows(), total=len(matched_df), desc="Processing matched peptides"):
#     # Get demodified peptide
#     peptide_demod = remove_modifications(row['Annotation'])
    
#     # Extract mass shift from the modified peptide
#     mod_matches = re.findall(r'([+-][\d.]+)', str(row['Annotation']))
#     mass_shift = mod_matches[0] if mod_matches else '+0'
    
#     # Get charge (assuming there's a 'Charge' column in matched_df)
#     charge = row.get('Charge', 0)
    
#     # Add to set as tuple
#     identified_peptide_list.add((peptide_demod, mass_shift, charge))

# print(f"\nCreated identified_peptide_list with {len(identified_peptide_list)} unique entries")
# print("\nFirst few entries:")
# for i, entry in enumerate(list(identified_peptide_list)):
#     print(entry)

def a fucntion for checking reason mismatch, and filling peptide type,  default is leave blank for mismatch_reason
    miss_3_cleavages ----Peptide has 3+ missed cleavages for trypsin
    lost_3_aa_Nterm--- Peptide lost 3+ AAs from the N-term (prefix) - this is the MinNTermAdd if its larger than 3
    Cterm_non_tryptic --- Peptide C-term (suffix/end) is non-Tryptic (i.e., no K/R at the end)

    for peptide_type column 
        read from peptide_length
        HLA1 -- Peptide length 8-12 = mark as HLA1
        HLA2 -- Peptide length 9-20 = mark as HLA2
        Otherwise mark as OtherLength


In [5]:
# def analyze_peptide_mismatches(df):
#     """
#     Function to analyze peptide mismatches and assign peptide types
#     """
#     def is_identified(peptide_demod, mass_shift, charge, identified_peptide_list):
#         """Check if the peptide is in the identified peptide list"""
#         return (peptide_demod, mass_shift, charge) in identified_peptide_list

#     def count_missed_cleavages(peptide_seq):
#         """Count missed cleavages for trypsin (K/R not at C-terminus)"""
#         if pd.isna(peptide_seq) or len(peptide_seq) == 0:
#             return 0
#         missed = 0
#         for aa in peptide_seq[:-1]:
#             if aa in ['K', 'R']:
#                 missed += 1
#         return missed

#     def is_cterm_tryptic(peptide_seq):
#         """Check if C-terminus is tryptic (ends with K or R)"""
#         if pd.isna(peptide_seq) or len(peptide_seq) == 0:
#             return False
#         return peptide_seq[-1] in ['K', 'R']

#     def check_HLA(length):
#         """Assign peptide type based on length"""
#         if 8 <= length <= 12:
#             return 'HLA1'
#         elif 13 <= length <= 24:
#             return 'HLA2'
#         else:
#             return 'Others'

#     def check_C57(peptide_seq):
#         """Check if at least one Cysteine carries a +57 modification"""
#         if pd.isna(peptide_seq) or len(peptide_seq) == 0:
#             return False
#         if 'C' not in peptide_seq:
#             return True
#         return bool(re.search(r'C\+57(?:\.\d+)?', peptide_seq))

#     def get_mismatch_reason(row):
#         is_ident = is_identified(
#             row['peptide_demod'],
#             row.get('mass_shift', '+0'),
#             row.get('Charge', 0),
#             identified_peptide_list
#         )
#         if is_ident:
#             return 'IDENTIFIED'

#         standard_mods = [1, 16, 42, 43, -17]
#         tolerance = 0.5
#         mod_matches = re.findall(r'([+-][\d.]+)', str(row['peptide']))
#         for mod_str in mod_matches:
#             mod_value = float(mod_str)
#             if not any(abs(mod_value - std_mod) <= tolerance for std_mod in standard_mods):
#                 return 'non_standard_modification'

#         mod_count = row['peptide'].count('+') + row['peptide'].count('-')
#         if mod_count >= 2:
#             return 'multiple_modifications'

#         if row['peptide_length'] > 40:
#             return 'peptide_length_>_40'

#         if is_cterm_tryptic(row['peptide_demod']):
#             if row['MinNTermAdd'] >= 3:
#                 return 'lost_3_aa_Nterm'
#             if count_missed_cleavages(row['peptide_demod']) >= 3:
#                 return 'miss_3_cleavages'
#             return 'regular_tryptic'

#         if 8 <= row['peptide_length'] <= 12:
#             return 'HLA1'
#         elif 13 <= row['peptide_length'] <= 24:
#             return 'HLA2'

#         if not check_C57(row['peptide']):
#             return 'C_without_+57_modification'

#         return 'Others'

#     if df.empty:
#         df['reason_mismatch'] = pd.Series(dtype='object')
#         return df

#     reasons = df.apply(get_mismatch_reason, axis=1)
#     if isinstance(reasons, pd.DataFrame):
#         if reasons.shape[1] == 1:
#             reasons = reasons.iloc[:, 0]
#         else:
#             raise ValueError("Mismatch reason assignment produced multiple columns")
#     df['reason_mismatch'] = reasons
#     return df

# # Apply the analysis to the summary dataframe
# summary_df = analyze_peptide_mismatches(summary_df)

# # Save the updated dataframe
# summary_df.to_csv('mismatch_summary.tsv', sep='\t', index=False)

# print(f"Updated mismatch_summary.tsv with peptide types and mismatch reasons")
# print("\nMismatch reason distribution:")
# print(summary_df['reason_mismatch'].value_counts())
# print("\nFirst few rows:")
# summary_df.head()

In [ ]:
# # Filter for rows where reason_mismatch is blank and peptide_type is OtherLength
# filtered_df = summary_df[(summary_df['reason_mismatch'] == 'Others') ]

# # Save the filtered data to a new TSV file
# filtered_df.to_csv('not_expected_mismatch.tsv', sep='\t', index=False)

# print(f"Created filtered_otherlength_nomismatch.tsv with {len(filtered_df)} rows")
# print(f"Total rows in original data: {len(summary_df)}")
# print(f"Filtered rows (blank reason_mismatch AND OtherLength): {len(filtered_df)}")
# print("\nFirst few rows of filtered data:")
# filtered_df.head()

: 